In [ ]:
# CUADERNO 22 · CELDA 1. Tablas y figuras del capítulo de resultados (solo lectura)
#   Lee los registros ya escritos de H1, H2, H3 y la lista del objetivo 6, comprueba sus huellas
#   y escribe tablas (CSV y un Excel con una hoja por tabla) y figuras (PNG) en una carpeta nueva
#   de C:\SERCOP_FINAL\capitulo_resultados\. No recalcula ninguna prueba, no abre la clave de H3,
#   no lee puntajes.parquet ni ningún RUC. No sobrescribe nada.
import hashlib, json, platform
from datetime import datetime
from pathlib import Path
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
COMA = FuncFormatter(lambda v, _: f"{v:g}".replace(".", ","))

M = Path(r"C:\SERCOP_FINAL\modelos")
H1D, H2D = M / "20261016_1207_h1final", M / "20261016_1845_h2_h3"
H3D, O6D = M / "20261121_1457_h3", M / "20261121_2235_objetivo6"
SALIDA = Path(r"C:\SERCOP_FINAL\capitulo_resultados")
ESPERADAS = {  # huellas ya anotadas en el registro de cifras y en las enmiendas
    H1D / "registro_h1.json": "adefee418dec29c5",
    H3D / "registro_h3.json": "3930dcd881399779",
    O6D / "lista_objetivo6.csv": "ad44ce9a570e0082",
}

def sha(r):
    h = hashlib.sha256()
    with open(r, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

HUELLAS = {}
for ruta in [H1D / "registro_h1.json", H1D / "h1_inyecciones.csv", H2D / "registro_h2.json",
             H3D / "registro_h3.json", H3D / "h3_procesos_final.csv",
             O6D / "lista_objetivo6.csv", O6D / "comparacion_listas_total.csv"]:
    HUELLAS[ruta.name] = sha(ruta)
    esperada = ESPERADAS.get(ruta)
    if esperada and not HUELLAS[ruta.name].startswith(esperada):
        raise SystemExit(f"{ruta.name} no tiene la huella registrada ({esperada}): para y avisa. No se escribió nada.")

DIR = SALIDA / datetime.now().strftime("%Y%m%d_%H%M")
DIR.mkdir(parents=True, exist_ok=False)
TABLAS, LINEAS = {}, []
def say(*a):
    t = " ".join(str(x) for x in a); print(t); LINEAS.append(t)
def tabla(nombre, df, titulo):
    TABLAS[nombre] = df
    df.to_csv(DIR / f"{nombre}.csv", index=False, encoding="utf-8-sig")
    say(f"\n{nombre}. {titulo}"); say(df.to_string(index=False))

METODOS = {"ensamble": "Ensamble", "base": "Índice simple", "if": "Isolation Forest",
           "lof": "LOF", "ae": "Autoencoder"}

# ---------- H1 ----------
R1 = json.loads((H1D / "registro_h1.json").read_text(encoding="utf-8"))["h1"]
tabla("T1_h1_confirmatoria", pd.DataFrame([dict(
    inyecciones=R1["inyecciones"], semillas=len(R1["semillas"]), grupos=len(R1["grupos"]),
    recuperacion_ensamble=round(R1["recuperacion_ensamble"], 4),
    recuperacion_indice_simple=round(R1["recuperacion_indice"], 4),
    D=round(R1["D"], 4), ic95_inf=round(R1["ic95"][0], 4), ic95_sup=round(R1["ic95"][1], 4),
    p_unilateral=R1["p_unilateral"], p_remuestreo_por_inyeccion=R1["sensibilidad_por_inyeccion_p"],
    decision="se cumple" if R1["cumple"] else "no se cumple")]),
    "H1: recuperación en el 1 % superior, ensamble contra índice simple (confirmatoria)")

I = pd.read_csv(H1D / "h1_inyecciones.csv")
t2 = (I.groupby("metodo")[["en_top_1", "en_top_2", "en_top_5"]].mean().round(4)
      .rename(columns={"en_top_1": "recuperacion_1pct", "en_top_2": "recuperacion_2pct", "en_top_5": "recuperacion_5pct"}))
t2 = t2.loc[list(METODOS)].reset_index().assign(metodo=lambda d: d["metodo"].map(METODOS))
t2.loc[len(t2)] = ["Azar (referencia)", 0.01, 0.02, 0.05]
tabla("T2_h1_por_metodo", t2, "H1, descriptivo: recuperación por método y corte (7.600 inyecciones, 20 semillas)")
t3 = I.pivot_table(index="grupo", columns="metodo", values="en_top_1", aggfunc="mean").round(3)
t3 = t3[list(METODOS)].rename(columns=METODOS).reset_index()
t3.insert(1, "inyecciones", I[I["metodo"] == "ensamble"].groupby("grupo").size().loc[t3["grupo"]].values)
tabla("T3_h1_por_grupo", t3, "H1, descriptivo: recuperación al 1 % por grupo y método")
t4 = I.pivot_table(index="tipo", columns="metodo", values="en_top_1", aggfunc="mean").round(3)
t4 = t4[list(METODOS)].rename(columns=METODOS).reset_index()
t4.insert(1, "inyecciones", I[I["metodo"] == "ensamble"].groupby("tipo").size().loc[t4["tipo"]].values)
tabla("T4_h1_por_tipo", t4, "H1, descriptivo: recuperación al 1 % por tipo de anomalía inyectada y método")

# ---------- H2 ----------
R2 = json.loads((H2D / "registro_h2.json").read_text(encoding="utf-8"))
t5 = pd.DataFrame(R2["h2"])[["variante", "estatus", "procesos", "contribuyentes", "T", "critico_95", "p_unilateral"]]
t5["T"], t5["critico_95"] = t5["T"].round(4), t5["critico_95"].round(4)
tabla("T5_h2", t5, "H2: mediana de percentiles de los procesos de contribuyentes del catastro del SRI (9.999 permutaciones)")

# ---------- H3 ----------
R3 = json.loads((H3D / "registro_h3.json").read_text(encoding="utf-8"))
h = R3["h3"]
fila = lambda lado, k, n, ic: dict(lado=lado, procesos=n, con_hallazgo=k, proporcion=round(k / n, 4),
                                    wilson95_inf=round(ic[0], 4), wilson95_sup=round(ic[1], 4))
t6 = pd.DataFrame([fila("Priorizados", *h["priorizados"], h["ic95_priorizados"]),
                   fila("Controles", *h["controles"], h["ic95_controles"])])
t6["diferencia_puntos"] = [round(100 * (t6.proporcion[0] - t6.proporcion[1]), 1), None]
t6["razon_odds"] = [round(h["razon_odds"], 2), None]
t6["p_fisher_unilateral"] = [round(h["p_fisher"], 4), None]
t6["kappa_24_dobles"] = [round(R3["kappa"], 3), None]
t6["acuerdo_simple"] = [round(R3["acuerdo"], 3), None]
t6["resultado"] = ["provisional" if R3["provisional"] else "final", None]
tabla("T6_h3", t6, "H3: procesos con hallazgo en la revisión documental ciega")
P3 = pd.read_csv(H3D / "h3_procesos_final.csv")
P3 = P3[P3["hallazgo"].notna()]
t7 = (P3.groupby(["grupo", "lado"]).agg(procesos=("hallazgo", "size"), con_hallazgo=("hallazgo", "sum"))
      .unstack("lado").fillna(0).astype(int))
t7.columns = [f"{a}_{b}" for a, b in t7.columns]
tabla("T7_h3_por_grupo", t7.reset_index(), "H3, descriptivo: procesos con hallazgo por grupo y lado")

# ---------- Resumen de hipótesis ----------
holm = R3["holm"]
t8 = pd.DataFrame([
    dict(hipotesis="H1 (primaria)", prueba="Diferencia de recuperación, remuestreo por semillas", p=R1["p_unilateral"],
         umbral=0.05, decision="se cumple" if R1["cumple"] else "no se cumple"),
    dict(hipotesis="H2 (secundaria)", prueba="Permutación por proveedor, Holm con H3", p=holm["p_H2"],
         umbral=0.025 if holm["p_H2"] <= holm["p_H3"] else 0.05, decision="se rechaza la nula" if holm["rechaza"]["H2"] else "no se rechaza la nula"),
    dict(hipotesis="H3 (secundaria)", prueba="Exacta de Fisher unilateral, Holm con H2", p=round(holm["p_H3"], 4),
         umbral=0.025 if holm["p_H3"] < holm["p_H2"] else 0.05, decision="se cumple" if holm["rechaza"]["H3"] else "no se cumple"),
])
tabla("T8_resumen_hipotesis", t8, "Resumen de las hipótesis preregistradas")
say("Decisión general (preregistro, sección 10): " + ("respaldada" if R1["cumple"] and (holm["rechaza"]["H2"] or holm["rechaza"]["H3"])
    else "parcialmente respaldada" if R1["cumple"] else "no respaldada; el resultado favorece al índice simple"))

# ---------- Objetivo 6 ----------
L = pd.read_csv(O6D / "lista_objetivo6.csv")
n = len(L)
t9 = (L.groupby("grupo").agg(procesos=("ocid", "size"), monto_mediano_usd=("monto_final", "median"),
                             monto_total_millones=("monto_final", lambda s: s.sum() / 1e6))
      .sort_values("procesos", ascending=False).reset_index())
t9["pct_de_la_lista"] = (100 * t9["procesos"] / n).round(1)
t9[["monto_mediano_usd", "monto_total_millones"]] = t9[["monto_mediano_usd", "monto_total_millones"]].round(2)
tabla("T9_lista_por_grupo", t9, f"Objetivo 6: lista principal ({n} procesos) por grupo")
t10 = L.groupby("anio_proc").size().rename("procesos").reset_index()
t10["pct_de_la_lista"] = (100 * t10["procesos"] / n).round(1)
tabla("T10_lista_por_anio", t10, "Objetivo 6: lista principal por año de inicio")
partes = (L[["ocid", "explica"]].dropna().assign(explica=lambda d: d["explica"].astype(str).str.replace(r"[;|/]", ",", regex=True).str.split(","))
          .explode("explica"))
partes["variable"] = partes["explica"].str.strip().str.split().str[0]
partes = partes[partes["variable"].notna() & (partes["variable"] != "")]
primera = partes.groupby("ocid", sort=False).head(1)
t11 = pd.DataFrame({"procesos_que_la_mencionan": partes.drop_duplicates(["ocid", "variable"])["variable"].value_counts(),
                    "procesos_donde_es_la_primera": primera["variable"].value_counts()}).fillna(0).astype(int)
t11 = t11.sort_values("procesos_que_la_mencionan", ascending=False).reset_index(names="variable")
t11["pct_de_la_lista"] = (100 * t11["procesos_que_la_mencionan"] / n).round(1)
tabla("T11_lista_variables", t11, "Objetivo 6: variables de mayor rareza que explican la lista (columna explica)")
t12 = pd.read_csv(O6D / "comparacion_listas_total.csv")
nombres = {"top_base": "Índice simple (1 %)", "top_if": "Isolation Forest (1 %)",
           "top_ensamble": "Ensamble antes de la regla (1 %)", "priorizado": "Lista principal"}
t12["lista_1"], t12["lista_2"] = t12["lista_1"].map(nombres), t12["lista_2"].map(nombres)
tabla("T12_comparacion_listas", t12, "Objetivo 6, descriptivo: procesos compartidos y Jaccard entre listas")

# ---------- Figuras ----------
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
f, ax = plt.subplots(figsize=(6.5, 3.6))
d = TABLAS["T2_h1_por_metodo"].iloc[:-1]
ax.bar(d["metodo"], d["recuperacion_1pct"], color="#4C72B0")
ax.axhline(0.01, ls="--", color="gray", lw=1, label="Azar (0,01)"); ax.legend(frameon=False, fontsize=8)
for i, v in enumerate(d["recuperacion_1pct"]):
    ax.text(i, v + 0.003, f"{v:.3f}".replace(".", ","), ha="center", fontsize=8)
ax.yaxis.set_major_formatter(COMA); ax.set_ylabel("Recuperación en el 1 % superior"); ax.set_title("H1: anomalías inyectadas recuperadas por método")
f.tight_layout(); f.savefig(DIR / "F1_h1_recuperacion.png", dpi=200); plt.close(f)

f, ax = plt.subplots(figsize=(5, 3.4))
for i, r in t6.iterrows():
    ax.errorbar(i, r["proporcion"], yerr=[[r["proporcion"] - r["wilson95_inf"]], [r["wilson95_sup"] - r["proporcion"]]],
                fmt="o", capsize=6, color="#4C72B0")
    ax.text(i + 0.08, r["proporcion"], f"{r['con_hallazgo']}/{r['procesos']}", va="center", fontsize=9)
ax.set_xticks([0, 1], t6["lado"]); ax.set_xlim(-0.5, 1.5); ax.set_ylim(0, None); ax.yaxis.set_major_formatter(COMA)
ax.set_ylabel("Proporción con hallazgo (IC 95 % de Wilson)")
ax.set_title(f"H3: revisión documental ciega (p = {h['p_fisher']:.2f})".replace(".", ","))
f.tight_layout(); f.savefig(DIR / "F2_h3_hallazgos.png", dpi=200); plt.close(f)

f, ax = plt.subplots(figsize=(6.5, 3.4))
ax.bar(t10["anio_proc"].astype(str), t10["procesos"], color="#4C72B0")
ax.set_ylabel("Procesos en la lista"); ax.set_title(f"Objetivo 6: lista principal por año de inicio ({n} procesos)")
f.tight_layout(); f.savefig(DIR / "F3_lista_por_anio.png", dpi=200); plt.close(f)

# ---------- Excel, registro y resumen ----------
with pd.ExcelWriter(DIR / "tablas_resultados.xlsx") as xw:
    for k, v in TABLAS.items():
        v.to_excel(xw, sheet_name=k[:31], index=False)
reg = dict(fecha=datetime.now().isoformat(timespec="seconds"), fuentes=HUELLAS,
           salidas={p.name: sha(p) for p in sorted(DIR.iterdir()) if p.is_file()},
           python=platform.python_version(), pandas=pd.__version__, matplotlib=matplotlib.__version__)
(DIR / "registro_tablas.json").write_text(json.dumps(reg, ensure_ascii=False, indent=1), encoding="utf-8")
say(f"\n{len(TABLAS)} tablas y 3 figuras en {DIR}")
(DIR / "resumen_para_chat.txt").write_text("\n".join(LINEAS), encoding="utf-8")
print("Pega esta salida en el chat (no trae RUC ni procesos uno por uno).")

In [ ]:
# CUADERNO 22 · CELDA 2. H3 con solo el revisor B (análisis exploratorio, posterior al resultado)
#   Motivo: el revisor A informó el 21-nov que no revisó sus procesos con exactitud. Este análisis
#   se decide después de conocer H3, no entra en Holm ni cambia ninguna decisión, y se informa
#   salga lo que salga. Usa los 72 procesos de la hoja del revisor B con la respuesta de B
#   (en los dos desacuerdos B había marcado hallazgo). Solo imprime conteos.
import hashlib, json, math
from datetime import datetime
from pathlib import Path
import pandas as pd

H3D = Path(r"C:\SERCOP_FINAL\modelos\20261121_1457_h3")
HOJA_B = Path(r"C:\SERCOP_FINAL\revision_h3_llenas\revisor_B.xlsx")
SALIDA = Path(r"C:\SERCOP_FINAL\capitulo_resultados")

def sha(r):
    return hashlib.sha256(Path(r).read_bytes()).hexdigest()
if not sha(HOJA_B).startswith("a838c0926df363cc"):
    raise SystemExit("revisor_B.xlsx no es la hoja registrada en H3: para y avisa.")
if not sha(H3D / "registro_h3.json").startswith("3930dcd881399779"):
    raise SystemExit("registro_h3.json no es el registrado: para y avisa.")

codigos_B = set(pd.read_excel(HOJA_B, sheet_name="revision", usecols=["codigo"])["codigo"].dropna().astype(str).str.strip())
F = pd.read_csv(H3D / "h3_procesos_final.csv")
F = F[F["final"].astype(str).isin(codigos_B) & F["hallazgo"].notna()].copy()
DESACUERDOS_B_HALLAZGO = {"R-042", "R-100"}   # en la celda 2 de H3, B marcó hallazgo y A no
F["hallazgo_B"] = F["hallazgo"].astype(int)
F.loc[F["final"].isin(DESACUERDOS_B_HALLAZGO), "hallazgo_B"] = 1
if len(F) != 72 or int(F["hallazgo_B"].sum()) != 9:
    raise SystemExit(f"Se esperaban 72 procesos y 9 hallazgos de B; salen {len(F)} y {int(F['hallazgo_B'].sum())}: para y avisa.")

def wilson(k, n, z=1.959964):
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n); r = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return (c - r) / d, (c + r) / d
def fisher_mayor(a, n1, c, n2):   # P(X >= a), hipergeométrica: misma prueba que H3
    K, N = a + c, n1 + n2
    return sum(math.comb(K, x) * math.comb(N - K, n1 - x) for x in range(a, min(K, n1) + 1)) / math.comb(N, n1)

g = F.groupby("lado")["hallazgo_B"].agg(["sum", "size"])
a, n1 = int(g.loc["priorizado", "sum"]), int(g.loc["priorizado", "size"])
c, n2 = int(g.loc["control", "sum"]), int(g.loc["control", "size"])
p = fisher_mayor(a, n1, c, n2)
(l1, u1), (l2, u2) = wilson(a, n1), wilson(c, n2)
filas = [dict(lado="Priorizados", procesos=n1, con_hallazgo=a, proporcion=round(a / n1, 4), wilson95_inf=round(l1, 4), wilson95_sup=round(u1, 4)),
         dict(lado="Controles", procesos=n2, con_hallazgo=c, proporcion=round(c / n2, 4), wilson95_inf=round(l2, 4), wilson95_sup=round(u2, 4))]
T = pd.DataFrame(filas)
T["p_fisher_unilateral"] = [round(p, 4), None]
T["estatus"] = ["exploratorio, posterior a H3, fuera de Holm", None]
print("H3 con solo el revisor B (exploratorio):")
print(T.to_string(index=False))
print(f"Diferencia {100 * (a / n1 - c / n2):+.1f} puntos | p unilateral de Fisher = {p:.4f}")

DIR = SALIDA / (datetime.now().strftime("%Y%m%d_%H%M") + "_h3_solo_B")
DIR.mkdir(parents=True, exist_ok=False)
T.to_csv(DIR / "T13_h3_solo_revisor_B.csv", index=False, encoding="utf-8-sig")
(DIR / "registro.json").write_text(json.dumps(dict(
    fecha=datetime.now().isoformat(timespec="seconds"), hoja_B=sha(HOJA_B), registro_h3=sha(H3D / "registro_h3.json"),
    h3_procesos_final=sha(H3D / "h3_procesos_final.csv"), priorizados=[a, n1], controles=[c, n2], p_fisher=p,
    motivo="El revisor A informó que no revisó con exactitud; análisis exploratorio decidido después de H3."),
    ensure_ascii=False, indent=1), encoding="utf-8")
print("Guardado en", DIR)